# Separate trading UI, recurring engine, and trace storage: practice

Read initial strategies → configure run interval/market policy → display database state in UI → repeatedly gather trader runs → store trace logs.

Use this with [the walkthrough](walkthrough.md).

**Source:**

- [6_mcp/5_lab5.ipynb](../../../../../6_mcp/5_lab5.ipynb)

**Run one cell at a time:** predict → run with Shift+Enter → explain → change one thing. Select the root project .venv kernel.

**Offline adaptation:** code below uses fictional data or local libraries. It never executes the linked source, reads a credential file, sends messages, starts a UI/server, or changes the course databases. Simulations are labeled; they do not verify external integrations.

## 1. A recurring engine is different from a UI

The UI displays state. An engine repeatedly invokes workers. A finite practice loop makes the scheduling visible without leaving a background job running.

**Predict:** How many worker invocations happen with two rounds and four traders?

In [ ]:
import asyncio
calls = []
async def trader_run(name):
    calls.append(name)
    await asyncio.sleep(0)
    return f"{name}: dry run"
for round_number in range(2):
    results = await asyncio.gather(*(trader_run(name) for name in ["A", "B", "C", "D"]))
    print(round_number + 1, results)
assert len(calls) == 8

**Check:** Eight. Real recurring runs can accumulate API usage and change stored account state.

**Change one thing:** Add a market-open boolean that skips a round. Explain why the source while True needs an intentional stop method.

## 2. Simulated accounts change state

Buying shares updates cash, holdings, and a transaction record. Practice against a dictionary so you do not import account modules that create or alter databases.

**Predict:** How much cash remains after buying three fictional shares at 100 each?

In [ ]:
account = {"cash": 1000.0, "holdings": {}, "transactions": []}
def buy(symbol, quantity, price):
    cost = quantity * price
    if quantity <= 0 or price <= 0 or cost > account["cash"]:
        raise ValueError("Invalid or unaffordable order")
    account["cash"] -= cost
    account["holdings"][symbol] = account["holdings"].get(symbol, 0) + quantity
    account["transactions"].append({"symbol": symbol, "quantity": quantity, "price": price})
buy("DEMO", 3, 100)
print(account)
assert account["cash"] == 700 and account["holdings"]["DEMO"] == 3

**Check:** 700. These are fictional values and this code places no real order.

**Change one thing:** Try an unaffordable order and confirm the balance is unchanged. Explain why reset is a data-changing action.

## 3. Cache and simulation are not live prices

The trading exercises use market data and cached values. A fixed fictional quote lets you observe caching. A fallback simulation must stay identifiable as simulated data.

**Predict:** How many provider-fixture calls occur for the same cached symbol?

In [ ]:
from functools import lru_cache
calls = []
@lru_cache(maxsize=8)
def practice_quote(symbol):
    calls.append(symbol)
    return {"symbol": symbol, "price": 100.0, "source": "fictional fixture"}
first = practice_quote("DEMO")
second = practice_quote("DEMO")
print(first, "Fixture calls:", len(calls))
assert len(calls) == 1 and second["source"] == "fictional fixture"

**Check:** One. Cached or random fallback prices are not verified current market values.

**Change one thing:** Call a second symbol. Explain how you would record a quote timestamp and expire old cache entries.

## 4. Configuration stays outside executable examples

Environment-variable names identify settings. Values belong in .environment/.env and are loaded deliberately. This exercise inspects only the path and simulated presence flags; it reads no private file or variable values.

**Predict:** Does finding a file prove that authentication works?

In [ ]:
from pathlib import Path
# Find the repository from a kernel started inside it.
root = next((p for p in (Path.cwd(), *Path.cwd().parents)
             if (p / "1_foundations" / "3_lab3.ipynb").is_file()), None)
if root is None:
    raise FileNotFoundError("Start your kernel inside ai_agents")
configuration_path = root / ".environment" / ".env"
print("Configuration path:", configuration_path)
simulated_presence = {"OPENAI_API_KEY": True, "GOOGLE_API_KEY": False}
print(simulated_presence)  # Booleans only; these are fictional flags.
assert configuration_path.name == ".env"

**Check:** No. File existence, variable presence, and successful provider authentication are separate checks.

**Change one thing:** Explain the difference between a missing file, a missing variable, and a provider rejecting authentication. Do not print token prefixes or values.

## Rebuild from memory

1. Recreate the main data-flow step in a new scratch cell.
2. Add one case that should fail and identify its exception/stop condition.
3. Compare your version with the linked source cells or functions.
4. Explain where a real model or service would enter, and what it could change.

For deliberate original-source execution, see the shared [configuration guide](../../../configuration.md). All provider/transport integrations remain UNVERIFIED at runtime.